# LC 198 — House Robber
**Difficulty:** Medium | **Pattern:** 1D Dynamic Programming

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> At each house you face exactly one
binary choice: rob it (add its money, skip the neighbour) or skip
it (carry forward the best seen so far). The optimal substructure
lets you collapse the whole array to just two rolling variables.
</div>

## Official Problem Statement

You are a professional robber planning to rob houses along a
street. Each house has a certain amount of money stashed. The
only constraint stopping you from robbing each of them is that
adjacent houses have security systems connected — **robbing two
adjacent houses will alert the police**.

Given an integer array `nums` representing the amount of money
in each house, return the **maximum amount** you can rob tonight
without alerting the police.

**Constraints:**
- `1 <= nums.length <= 100`
- `0 <= nums[i] <= 400`

## What This Is Actually Asking

Pick a subset of elements from an array such that no two picked
elements are adjacent, and the sum is maximised. The "no adjacent"
rule creates the classic DP dependency: if you rob house `i` you
must skip `i-1`, so the best you can do at `i` is either skip it
(keep the previous best) or rob it (add `nums[i]` to the best
two steps back). Two rolling variables capture this fully.

## Walk Through an Example by Hand

`nums = [2, 7, 9, 3, 1]`

```
i=0: rob 2              → prev2=0, prev1=2
i=1: max(2, 0+7)=7      → prev2=2, prev1=7
i=2: max(7, 2+9)=11     → prev2=7, prev1=11
i=3: max(11, 7+3)=11    → prev2=11, prev1=11
i=4: max(11, 11+1)=12   → prev2=11, prev1=12
```

Answer = **12** (rob houses 0, 2, 4 → 2+9+1=12).

## The Picture

```
nums =  [ 2,  7,  9,  3,  1 ]
index     0   1   2   3   4

dp[i] = max(dp[i-1],  dp[i-2] + nums[i])
         skip house i   rob house i

dp:     [ 2,  7, 11, 11, 12 ]

Rolling window:
  prev2   prev1   cur
    0   →   2
    2   →   7
    7   →  11
   11   →  11
   11   →  12   ← answer

Recurrence:
  dp[0] = nums[0]
  dp[1] = max(nums[0], nums[1])
  dp[i] = max(dp[i-1], dp[i-2] + nums[i])
```

## When To Use This Pattern

- When elements cannot be selected if they are **adjacent**,
  think **House Robber DP**.
- When the optimal choice at index `i` depends only on `i-1`
  and `i-2`, think **two-variable rolling DP**.
- When the array is linear (not circular), think **single
  left-to-right sweep**.
- When asked for maximum subset sum with a spacing constraint,
  think **this exact recurrence**.
- When n ≤ 100, even O(n) space is trivially fine, but O(1)
  rolling demonstrates clean understanding.

## The Approach

Maintain two variables, `prev2` (best excluding last house) and
`prev1` (best including or excluding last house). At each new
house, compute `cur = max(prev1, prev2 + nums[i])`. Then slide:
`prev2 = prev1`, `prev1 = cur`. After the sweep, `prev1` holds
the answer.

In [1]:
from typing import List

In [2]:
def test_harness(func):
    cases = [
        # (nums, expected)
        ([1, 2, 3, 1],              4),
        ([2, 7, 9, 3, 1],          12),
        ([0],                        0),   # single house
        ([5],                        5),   # single house
        ([1, 2],                     2),   # two houses
        ([2, 1, 1, 2],               4),   # rob first & last
        ([0, 0, 0],                  0),   # all zeros
    ]
    passed = 0
    for nums, expected in cases:
        result = func(nums)
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        else:
            print(
                f"  {status}: nums={nums} "
                f"=> got {result}, want {expected}"
            )
    print(f"\nSummary: {passed}/{len(cases)} passed")

In [4]:
def rob(nums: List[int]) -> int:
    """
    Return max money robbable with no two adjacent houses.

    Strategy: rolling DP with two variables.
      prev2 = best result two steps back
      prev1 = best result one step back
      cur   = max(prev1, prev2 + nums[i])

    Args:
        nums: list of non-negative integers, len >= 1
    Returns:
        Maximum loot without alerting police.
    """
    if len(nums) == 1 :return (nums[0])
    if len(nums) == 2 :return (max(nums[0], nums[1]))
    cur , prev, pprev = 0, max(nums[0], nums[1]), nums[0]
    for i in range (2, len(nums)):
        cur = max(nums[i]+pprev , prev)
        pprev = prev
        prev = cur
    return cur
        




def test():

    # Quick debug — run this cell while building
    print(rob([1,2,3,1]))      # 4
    print(rob([2,7,9,3,1]))    # 12
    print(rob([5]))             # 5
    print(rob([2,1,1,2]))      # 4
    test_harness(rob)
test()


4
12
5
4

Summary: 7/7 passed


In [ ]:
# Uncomment and run when solution is ready
# test_harness(rob)

## Complexity

| Approach | Time | Space | Notes |
|---|---|---|---|
| Brute-force (all subsets) | O(2^n) | O(n) | Exponential |
| Memoised recursion | O(n) | O(n) | Top-down DP |
| DP array | O(n) | O(n) | Clean but wasteful |
| **Rolling two vars** | **O(n)** | **O(1)** | Optimal |

## Real World Connection

At **Citi**, risk models sometimes prohibit booking two
correlated trades back-to-back; maximising P&L under that
adjacency constraint is structurally identical to House Robber.
In **AWS Lambda** cost optimisation, certain warm-up invocations
must be spaced apart; choosing which batches to trigger for
maximum throughput uses the same recurrence. For a **data
engineer** scheduling ETL jobs with cooldown periods, maximising
data freshness subject to "no back-to-back heavy jobs" maps
directly to this DP. Understanding this pattern unlocks a whole
family of interval-scheduling problems.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra